# EDA of HAM10000

Exploratory data analysis of the training dataset, read from the common data registry.

**Before running:** build the registry and the split (`neo-data registry ham10000`, `neo-data split ham10000`).
Run all cells from top to bottom. Charts are saved to `docs/figures/` as SVG and PNG.

This notebook is committed **without outputs**: the sample images at the end are under the
HAM10000 licence (CC BY-NC 4.0) and must not be copied into the repository.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

import neo
from neo import paths, plots
from neo.data import registry, split

FIGURES = Path(neo.__file__).resolve().parents[2] / "docs" / "figures"
frame = registry.load("ham10000")
parts = split.load("ham10000")
print(
    f"{len(frame)} images, {frame['lesion_id'].nunique()} lesions, {frame['diagnosis'].nunique()} classes"
)
frame.head(3).T

## 1. Classes and imbalance
How many images per class? A strong imbalance means plain accuracy is misleading:
we report **balanced accuracy** instead.

In [ ]:
names = {k: f"{k} – {v.split(' (')[0]}" for k, v in registry.DIAGNOSES.items()}
counts = frame["diagnosis"].map(names).value_counts()
fig = plots.bar_counts(
    counts, "HAM10000: images per class", f"{len(frame):,} dermoscopic images; nevi are two thirds"
)
plots.save(fig, FIGURES, "eda_ham10000_classes")
print("largest / smallest class:", round(counts.max() / counts.min(), 1))

## 2. Images per lesion
Several images can show the same lesion. This is why the split is made **by lesion**.

In [ ]:
per_lesion = frame.groupby("lesion_id").size()
table = per_lesion.value_counts().sort_index()
table.index = [f"{n} image" + ("s" if n > 1 else "") for n in table.index]
fig = plots.bar_counts(
    table.iloc[::-1],
    "HAM10000: images per lesion",
    f"{len(per_lesion):,} lesions; {(per_lesion > 1).mean():.0%} have more than one image",
)
plots.save(fig, FIGURES, "eda_ham10000_images_per_lesion")
print("images that share their lesion with another image:", int(per_lesion[per_lesion > 1].sum()))

## 3. How reliable are the labels?
`label_source` says how each diagnosis was established. Histology is the strongest.

In [ ]:
order = ["histology", "follow_up", "consensus", "confocal"]
by_class = pd.crosstab(frame["diagnosis"], frame["label_source"])[order]
by_class = by_class.loc[frame["diagnosis"].value_counts().index]
fig = plots.stacked_share(
    by_class,
    "HAM10000: how each class was diagnosed",
    "Share of images per class; malignant classes are almost all histology",
)
plots.save(fig, FIGURES, "eda_ham10000_label_source")
by_class.assign(total=by_class.sum(axis=1))

## 4. Where do the images come from?
Four sub-collections: three from Vienna (Austria), one from Queensland (Australia).
If a class comes mostly from one centre, the model can learn the centre instead of the lesion.

In [ ]:
centres = ["vidir_molemax", "vidir_modern", "rosendahl", "vienna_dias"]
by_centre = pd.crosstab(frame["diagnosis"], frame["centre"])[centres]
by_centre = by_centre.loc[frame["diagnosis"].value_counts().index]
fig = plots.stacked_share(
    by_centre,
    "HAM10000: source of each class",
    "Share of images per class; vidir_molemax contains almost only nevi",
)
plots.save(fig, FIGURES, "eda_ham10000_centres")
by_centre.assign(total=by_centre.sum(axis=1))

## 5. Patients: age, sex, anatomical site, missing values

In [ ]:
missing = pd.DataFrame(
    {
        "missing": [
            frame["age"].isna().sum(),
            (frame["sex"] == "unknown").sum(),
            (frame["site"] == "unknown").sum(),
            frame["patient_id"].isna().sum(),
            frame["fitzpatrick"].isna().sum(),
        ],
    },
    index=["age", "sex", "site", "patient_id", "fitzpatrick"],
)
missing["share"] = (missing["missing"] / len(frame)).map("{:.1%}".format)
display(missing)
display(
    frame.groupby("diagnosis")["age"].describe()[["count", "mean", "25%", "50%", "75%"]].round(1)
)
display(pd.crosstab(frame["diagnosis"], frame["sex"], normalize="index").round(2))
fig = plots.bar_counts(
    frame["site"].value_counts(),
    "HAM10000: anatomical site",
    "Acral sites (hand, foot, acral) are rare",
)
plots.save(fig, FIGURES, "eda_ham10000_sites")

## 6. Malignancy
`malignant` is derived from the diagnosis: `mel`, `bcc`, `akiec`.

In [ ]:
print(f"malignant images: {frame['malignant'].sum()} ({frame['malignant'].mean():.1%})")
pd.crosstab(frame["diagnosis"], frame["malignant"])

## 7. Duplicates
Result of `neo-data duplicates ham10000`. Pairs across different lesions would leak between training and test.

In [ ]:
pairs_file = paths.subdir("registry") / "ham10000_duplicates.csv"
pairs = pd.read_csv(pairs_file)
print(
    f"{len(pairs)} pairs; identical files: {pairs['exact'].sum()}; across lesions: {(~pairs['same_lesion']).sum()}"
)
pairs.head()

## 7b. Artifacts
Result of `neo-data artifacts ham10000`.

- **Dark corners** are measured on the pixels (share of dark pixels in the four corners): reliable.
- **Hair** uses the MONET concept score with a threshold of 0.7: approximate.
- MONET scores for ink, ruler, sticker and bubbles did **not** pass the check by eye on dermoscopic
  images and are not used.

If an artifact is more frequent in one class, a model can use it as a shortcut.

In [ ]:
from neo.data import artifacts

scores = pd.read_csv(paths.subdir("registry") / "ham10000_artifact_scores.csv")
audit = frame.merge(scores, on="uid")
audit["dark_corners"] = audit["dark_corners_pixels"] > artifacts.DEFAULT_THRESHOLD
audit["hair"] = audit["hair"] >= 0.7

by_class = audit.groupby("diagnosis")["dark_corners"].agg(["mean", "size"])
fig = plots.bar_shares(
    by_class["mean"],
    by_class["size"],
    "HAM10000: images with dark corners, per class",
    "Round dermoscope field of view; twice as frequent in malignant images",
)
plots.save(fig, FIGURES, "eda_ham10000_dark_corners")

display(audit.groupby("centre")[["dark_corners", "hair"]].agg(["sum", "mean"]).round(3))
display(audit.groupby("malignant")[["dark_corners", "hair"]].mean().round(3))
display(audit.groupby(["centre", "malignant"])["dark_corners"].agg(["size", "mean"]).round(3))

## 8. The split by lesion (70 / 15 / 15)
Published in `ml/splits/ham10000_lesion_v1.csv`; never regenerated.

In [ ]:
split.check(parts, frame)
display(split.summary(parts, frame))
merged = parts.merge(frame, on="uid")
display(pd.crosstab(merged["split"], merged["malignant"], normalize="index").round(3))
display(pd.crosstab(merged["split"], merged["centre"], normalize="index").round(3))

## 9. A look at the images
Six random images per class. **Displayed only, not saved.**

In [ ]:
root = paths.data_dir()
classes = list(registry.DIAGNOSES)
fig, axes = plt.subplots(len(classes), 6, figsize=(12, 1.6 * len(classes)))
for row, diagnosis in enumerate(classes):
    sample = frame[frame["diagnosis"] == diagnosis].sample(6, random_state=0)
    for col, file in enumerate(sample["file"]):
        axes[row, col].imshow(Image.open(root / file))
        axes[row, col].axis("off")
    axes[row, 0].set_title(diagnosis, loc="left", fontsize=10)
fig.tight_layout()